# House Prices - Advanced Regression Techniques
##### Predict sales prices and practice feature engineering, RFs, and gradient boosting

##### It is your job to predict the sales price for each house. For each Id in the test set, you must predict the value of the SalePrice variable. 

### Importing Modules 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

##### Reading file from path( folder)

In [ ]:
foder_path = 'C:/Users/Ryzen7/Downloads/House competition/train.csv'

In [ ]:
data = pd.read_csv(foder_path)

### EDA
##### Data Inspection

In [ ]:
data.sample(3)

In [ ]:
data.info()

##### Filling missing Values. For the categorical features, I have been adviced the missing values are N/A(i.e not all houses have fenced, so the input is N/A). In this case, I have filled them with 'None'.

In [ ]:
data['MiscFeature'].fillna('None', inplace = True)

In [ ]:
data['MiscFeature'].value_counts()

In [ ]:
data['Fence'].fillna('None', inplace = True)

In [ ]:
data['Fence'].value_counts()

In [ ]:
data['PoolQC'].fillna('None', inplace = True)

In [ ]:
data['PoolQC'].value_counts()

In [ ]:
data['GarageCond'].fillna('None', inplace = True)

In [ ]:
data['GarageCond'].value_counts()

In [ ]:
data['GarageQual'].fillna('None', inplace = True)

In [ ]:
data['GarageQual'].value_counts()

In [ ]:
data['GarageFinish'].fillna('None', inplace = True)

In [ ]:
data['GarageFinish'].value_counts()

In [ ]:
data['GarageType'].fillna('None', inplace = True)

In [ ]:
data['GarageType'].value_counts()

In [ ]:
data['FireplaceQu'].fillna('None', inplace = True)

In [ ]:
data['FireplaceQu'].value_counts()

In [ ]:
data['Electrical'].value_counts()

In [ ]:
data['BsmtFinType2'].fillna('None', inplace = True)

In [ ]:
data['BsmtFinType2'].value_counts()

In [ ]:
data['BsmtFinType1'].fillna('None', inplace = True)

In [ ]:
data['BsmtFinType1'].value_counts()

In [ ]:
data['BsmtExposure'].fillna('None', inplace = True)

In [ ]:
data['BsmtExposure'].value_counts()

In [ ]:
data['BsmtCond'].fillna('None', inplace = True)

In [ ]:
data['BsmtCond'].value_counts()

In [ ]:
data['BsmtQual'].fillna('None', inplace = True)

In [ ]:
data['BsmtQual'].value_counts()

In [ ]:
data['MasVnrType'].fillna('None', inplace = True)

In [ ]:
data['MasVnrType'].value_counts()

In [ ]:
data['Alley'].fillna('None', inplace = True)

In [ ]:
data['Alley'].value_counts()

##### For the garage year built, there was 81 missing values, to fill those values I had to be sure the house actually have a garage.

In [ ]:
data[['GarageYrBlt', 'GarageType']].sample(5)

In [ ]:
data[data['GarageYrBlt'].isnull()]['GarageType'].value_counts(dropna = False)

##### From the above result, we can see the 81 missing values are due to house not have a garage, so i filled it with 0. But i have to do some feature engineering using indicators to avoid my model reading 0 as very old or something strong that will fault the performance of the model. With the feature engineering, the model will get a stronger training and understanding of the data and importance. 

In [ ]:
data['GarageYrBlt'].fillna(0, inplace = True)

##### Feature Engineering

In [ ]:
data['HasGarage'] = data['GarageYrBlt'].apply(
    lambda x: 1 if x > 0 else 0
)

In [ ]:
data['HasGarage'].value_counts()

##### In filling missing values for Masonry veneer area, I checked if the house have a masonry

In [ ]:
data[data['MasVnrArea'].isnull()]['MasVnrType'].value_counts(dropna = False)

##### I found out that the 8 missing values in MasVnrArea have no Masonry, so i filled that with 0. To be safer with the prediction, I will add a feature engineering using indicator for Has Masonry. This will create a more predicting power for the model

In [ ]:
data['MasVnrArea'].fillna(0, inplace = True)

##### Feature Engineering

In [ ]:
data['HasMansonry'] = (data['MasVnrArea'] > 0).astype(int)

In [ ]:
data[data['Electrical'].isnull()]

##### Electrical column have 1 missing value, using the assumption that all house have electricity, I will use most frequent occuring value to fill the missing one.

In [ ]:
data['Electrical'].fillna(data['Electrical'].mode()[0], inplace = True)

##### LotFrontage have over 200 missing values out of 1460 values. In other to fill this missing values, I have the check the features with a strong correlation with LotFrontage. Using the features strongly correlated, I will use LinearRegression model to fill the missing values.

In [ ]:
data[['LotFrontage', 'LotArea']].corr()

In [ ]:
numeric_only = data.select_dtypes(include = ['int64', 'float64'])

In [ ]:
corr_lot = numeric_only.corr()

In [ ]:
lotfrontage_corr = corr_lot['LotFrontage']
strong_corr = lotfrontage_corr[abs(lotfrontage_corr) > 0.40].sort_values(ascending=False)

In [ ]:
strong_corr

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
known = data[data['LotFrontage'].notna()]
missing = data[data['LotFrontage'].isna()]
features = ['LotArea', '1stFlrSF', 'GrLivArea']

model = LinearRegression()
model.fit(known[features], known['LotFrontage'])

data.loc[data['LotFrontage'].isna(), 'LotFrontage'] = \
    model.predict(missing[features])

In [ ]:
data['LotFrontage'].isna().sum()

##### Duplicating my data so I dont loose the original one

In [ ]:
train_data = data.copy()

In [ ]:
train_data.isna().sum()

In [ ]:
train_data.duplicated().sum()

In [ ]:
train_data['Id'].nunique()

In [ ]:
train_data.shape

In [ ]:
train_data.describe().round(2)

##### Data Distribution Visualisation

In [ ]:
train_data.hist(figsize=(20, 10), grid=False, color='skyblue', edgecolor='black')
plt.subplots_adjust(hspace=2.0, wspace=1.0)  # More space between subplots
plt.suptitle("House Data Histograms", fontsize=20, fontweight='bold')
plt.show()

In [ ]:
train_data[['MasVnrArea', 'HasMansonry']].sample(4)

##### Numeric Data Correlation

In [ ]:
corr = train_data.corr(numeric_only = True).round(3)
plt.figure(figsize = (20,9))
sns.heatmap(corr, annot = True);

In [ ]:
train_data.info()

##### Feature Engineering 

In [ ]:
train_data['HouseAge'] = train_data['YrSold'] - train_data['YearBuilt']

In [ ]:
train_data['HouseAge'].sample(3)

In [ ]:
train_data.loc[train_data['Id'].isin([766, 1050]), ['HouseAge', 'YrSold', 'YearBuilt']]

In [ ]:
train_data['TotalSquareFootage'] = train_data['1stFlrSF'] + train_data['2ndFlrSF'] + train_data['TotalBsmtSF']

In [ ]:
train_data['TotalSquareFootage'].sample(3)

##### Mapping The Ordinal Categorical Values

In [ ]:
train_data['LotShape'].value_counts()

In [ ]:
LotShape_mapping = {'Reg' : 0, 'IR1' : 1, 'IR2' : 2, 'IR3' : 3}

In [ ]:
train_data['LotShape'] = train_data['LotShape'].map(LotShape_mapping)

In [ ]:
train_data['LotShape'].value_counts()

In [ ]:
train_data['LandSlope'].value_counts()

In [ ]:
LandSlope_mapping = {'Gtl' : 0, 'Mod' : 1, 'Sev' : 2}

In [ ]:
train_data['LandSlope'] = train_data['LandSlope'].map(LandSlope_mapping)

In [ ]:
train_data['LandSlope'].value_counts()

In [ ]:
train_data['ExterQual'].value_counts()

In [ ]:
ExterQual_mapping =  {'Po' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}

In [ ]:
train_data['ExterQual'] = train_data['ExterQual'].map(ExterQual_mapping )

In [ ]:
train_data['ExterQual'].value_counts()

In [ ]:
train_data['ExterCond'].value_counts()

In [ ]:
ExterCond_mapping = {'Po' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}

In [ ]:
train_data['ExterCond'] = train_data['ExterCond'].map(ExterCond_mapping)

In [ ]:
train_data['ExterCond'].value_counts()

In [ ]:
train_data['BsmtQual'].value_counts()

In [ ]:
BsmtQual_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}

In [ ]:
train_data['BsmtQual'] = train_data['BsmtQual'].map(BsmtQual_mapping)

In [ ]:
train_data['BsmtQual'].value_counts()

In [ ]:
train_data['BsmtCond'].value_counts()

In [ ]:
BsmtCond_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}

In [ ]:
train_data['BsmtCond'] = train_data['BsmtCond'].map(BsmtCond_mapping)

In [ ]:
train_data['BsmtCond'].value_counts()

In [ ]:
train_data['BsmtExposure'].value_counts()

In [ ]:
BsmtExposure_mapping = {'None' : 0, 'No' : 1, 'Mn' : 2, 'Av' : 3, 'Gd' : 4}

In [ ]:
train_data['BsmtExposure'] = train_data['BsmtExposure'].map(BsmtExposure_mapping)

In [ ]:
train_data['BsmtExposure'].value_counts()

In [ ]:
train_data['BsmtFinType1'].value_counts()

In [ ]:
BsmtFinType1_mapping = {'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6}

In [ ]:
train_data['BsmtFinType1'] = train_data['BsmtFinType1'].map(BsmtFinType1_mapping)

In [ ]:
train_data['BsmtFinType1'].value_counts()

In [ ]:
train_data['BsmtFinType2'].value_counts()

In [ ]:
BsmtFinType2_mapping = {'GLQ':6, 'ALQ':5, 'BLQ':4, 'Rec':3, 'LwQ':2, 'Unf':1, 'None':0}

In [ ]:
train_data['BsmtFinType2'] = train_data['BsmtFinType2'].map(BsmtFinType2_mapping)

In [ ]:
train_data['BsmtFinType2'].value_counts()

In [ ]:
train_data['HeatingQC'].value_counts()

In [ ]:
HeatingQC_mapping = {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1, 'Po':0}

In [ ]:
train_data['HeatingQC'] = train_data['HeatingQC'].map(HeatingQC_mapping)

In [ ]:
train_data['HeatingQC'].value_counts()

In [ ]:
train_data['CentralAir'].value_counts()

In [ ]:
CentralAir_mapping = { 'Y':1, 'N':0}

In [ ]:
train_data['CentralAir'] = train_data['CentralAir'].map(CentralAir_mapping)

In [ ]:
train_data['CentralAir'].value_counts()

In [ ]:
train_data['KitchenQual'].value_counts()

In [ ]:
KitchenQual_mapping = {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1, 'Po':0}

In [ ]:
train_data['KitchenQual'] = train_data['KitchenQual'].map(KitchenQual_mapping)

In [ ]:
train_data['KitchenQual'].value_counts()

In [ ]:
train_data['FireplaceQu'].value_counts()

In [ ]:
FireplaceQu_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}

In [ ]:
train_data['FireplaceQu'] = train_data['FireplaceQu'].map(FireplaceQu_mapping)

In [ ]:
train_data['FireplaceQu'].value_counts()

In [ ]:
train_data['GarageFinish'].value_counts()

In [ ]:
GarageFinish_mapping = {'None':0, 'Unf':1, 'RFn':2, 'Fin':3}

In [ ]:
train_data['GarageFinish'] = train_data['GarageFinish'].map(GarageFinish_mapping)

In [ ]:
train_data['GarageFinish'].value_counts()

In [ ]:
train_data['GarageQual'].value_counts()

In [ ]:
GarageQual_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}

In [ ]:
train_data['GarageQual'] = train_data['GarageQual'].map(GarageQual_mapping)

In [ ]:
train_data['GarageQual'].value_counts()

In [ ]:
train_data['GarageCond'].value_counts()

In [ ]:
GarageCond_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}

In [ ]:
train_data['GarageCond'] = train_data['GarageCond'].map(GarageCond_mapping)

In [ ]:
train_data['GarageCond'].value_counts()

In [ ]:
train_data['PavedDrive'].value_counts()

In [ ]:
PavedDrive_mapping = { 'N':0, 'P':1, 'Y':2}

In [ ]:
train_data['PavedDrive'] = train_data['PavedDrive'].map(PavedDrive_mapping)

In [ ]:
train_data['PavedDrive'].value_counts()

In [ ]:
train_data['PoolQC'].value_counts()

In [ ]:
PoolQC_mapping = {'None' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}

In [ ]:
train_data['PoolQC'] = train_data['PoolQC'].map(PoolQC_mapping)

In [ ]:
train_data['PoolQC'].value_counts()

In [ ]:
train_data['Fence'].value_counts()

In [ ]:
Fence_mapping = {'None':0, 'MnWw':1, 'GdWo':2, 'MnPrv':3, 'GdPrv':4}

In [ ]:
train_data['Fence'] = train_data['Fence'].map(Fence_mapping)

In [ ]:
train_data['Fence'].value_counts()

In [ ]:
train_data.info()

In [ ]:
train_data['HasMansonry'].value_counts()

##### Renaming Features with numeric characters to alphabet so model can process them

In [ ]:
finalTrain_data = train_data.rename(columns = {"1stFlrSF" : "FirstFlrSF",
                                          "2ndFlrSF" : "SecondFlrSF",
                                          "3SsnPorch" : "ThreeSsnPorch"
                                         })

In [ ]:
Cat_column = finalTrain_data.select_dtypes(include ='object')

In [ ]:
Numeric_column = finalTrain_data.select_dtypes(include = ['int64', 'int32', 'float64'])

In [ ]:
corr_matrix = Numeric_column.corr()

##### Checking Features with correlation to the SalePrice

In [ ]:
sale_corr = corr_matrix['SalePrice']

In [ ]:
strong_features = sale_corr[abs(sale_corr) > 0.40] \
                    .sort_values(ascending=False)

print(strong_features)

In [ ]:
strong_features_main = strong_features.drop(['SalePrice']).index.tolist()

##### Predicting Actual Price Using Ordinary Least Square Regression

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
features = " + ".join(strong_features_main)

formula = f"SalePrice ~ {features}"

model1 = smf.ols(formula, data=finalTrain_data).fit()

print(model1.summary())

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = model1.model.exog
vif = [variance_inflation_factor(X, i) for i in range(X.shape[1])]

In [ ]:
y_pred = model1.predict(finalTrain_data)
y_true = finalTrain_data['SalePrice']

In [ ]:
Tab = pd.DataFrame({'actual' : y_true, 'Predictions': y_pred})

In [ ]:
print(Tab[:20])

##### Evaluating the model performance 

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

##### Visual of actual VS predicted 

In [ ]:
plt.figure(figsize=(6,4))

# Scatter plot with color gradient based on error
errors = y_true - y_pred
plt.scatter(y_true, y_pred, c=errors, cmap="coolwarm", alpha=0.6)

# Perfect prediction line
min_val = min(y_true.min(), y_pred.min())
max_val = max(y_true.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color="black", linewidth=2)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.colorbar(label="Prediction Error")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

sns.scatterplot(x=y_true, y=y_pred, hue=(y_true - y_pred), palette="coolwarm")

# Perfect line
plt.plot([y_true.min(), y_true.max()],
         [y_true.min(), y_true.max()],
         color="black")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.legend(title="Error")
plt.show()

##### Categorical Column Transformation 

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [ ]:
categorical_cols = finalTrain_data.select_dtypes(include=["object"]).columns
numeric_cols = finalTrain_data.select_dtypes(include=["int64", "int32","float64"]).columns
numeric_cols = numeric_cols.drop(["SalePrice", "Id"])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

##### Training RandomForest Model 

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

In [ ]:
X = finalTrain_data.drop(["SalePrice", "Id"], axis=1)
y = finalTrain_data["SalePrice"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model.fit(X_train, y_train)
rf_train_score = rf_model.score(X_train, y_train)
rf_test_score = rf_model.score(X_test, y_test)
print(f'The train score is :{rf_train_score:.3f}, and the test score is :{rf_test_score:.3f}')

In [ ]:
y_train_pred = rf_model.predict(X_train)
y_test_pred  = rf_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
print(Top_train[:10])

In [ ]:
print(Top_test[:10])

##### Validating the performane of RandomForect 

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

In [ ]:
plt.figure(figsize=(6,4))

sns.scatterplot(x=y_train, y=y_train_pred, hue=(y_train - y_train_pred), palette="coolwarm")

# Perfect line
plt.plot([y_train.min(), y_train.max()],
         [y_train.min(), y_train.max()],
         color="black")

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.legend(title="Error")
plt.show()

##### Feature importance of RandomForest model

In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

importances = rf_model.named_steps["model"].feature_importances_

importances_df = (pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False))

importances_df.head(20)

In [ ]:
new_features = importances_df.head(20)

In [ ]:
selected_features = new_features['feature'].tolist()

##### Retraining the model with the reduced features 

In [ ]:
selected_features = [col.replace("num__", "") for col in selected_features]

X = finalTrain_data[selected_features]
y = finalTrain_data["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)
rf_train_score = rf_model.score(X_train, y_train)
rf_test_score = rf_model.score(X_test, y_test)
print(f'The train score is :{rf_train_score:.3f}, and the test score is :{rf_test_score:.3f}')

In [ ]:
y_train_pred = rf_model.predict(X_train)
y_test_pred  = rf_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

##### Training XGBoost Model with the complete features 

In [ ]:
from xgboost import XGBRegressor

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ))
])

In [ ]:
X = finalTrain_data.drop(["SalePrice", "Id"], axis=1)
y = finalTrain_data["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
xgb_model.fit(X_train, y_train)
xgb_train_score = xgb_model.score(X_train, y_train)
xgb_test_score = xgb_model.score(X_test, y_test)
print(f'The train score is :{xgb_train_score:.3f}, and the test score is :{xgb_test_score:.3f}')

In [ ]:
y_train_pred = xgb_model.predict(X_train)
y_test_pred  = xgb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

In [ ]:
import lightgbm as lgb
import catboost as ctb

##### Training Lightgbm Model with the complete features 

In [ ]:
lgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22
))
])

In [ ]:
X = finalTrain_data.drop(["SalePrice", "Id"], axis=1)
y = finalTrain_data["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lgb_model.fit(X_train, y_train)
lgb_train_score = lgb_model.score(X_train, y_train)
lgb_test_score = lgb_model.score(X_test, y_test)
print(f'The train score is :{lgb_train_score:.3f}, and the test score is :{lgb_test_score:.3f}')

In [ ]:
y_train_pred = lgb_model.predict(X_train)
y_test_pred  = lgb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

##### Training Catboost Model with the complete features 

In [ ]:
ctb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ctb.CatBoostRegressor(
    iterations=100,  # Number of trees
    learning_rate=0.1,
    depth=3,
    verbose=200,  
    random_state=22
))
])

In [ ]:
X = finalTrain_data.drop(["SalePrice", "Id"], axis=1)
y = finalTrain_data["SalePrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
ctb_model.fit(X_train, y_train)
ctb_train_score = ctb_model.score(X_train, y_train)
ctb_test_score = ctb_model.score(X_test, y_test)
print(f'The train score is :{ctb_train_score:.3f}, and the test score is :{ctb_test_score:.3f}')

In [ ]:
y_train_pred = ctb_model.predict(X_train)
y_test_pred  = ctb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae = mean_absolute_error(y_train, y_train_pred)
r2 = r2_score(y_train, y_train_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

In [ ]:
finalTrain_data['SalesPrice'] = np.log1p(finalTrain_data['SalePrice'])

In [ ]:
finalTrain_data.sample(3)

##### SalePrice is heavily skewwed, I will use log to transform and balance the Distribution

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].hist(finalTrain_data['SalePrice'], color='skyblue', edgecolor='black')
ax[0].set_title('House Sale Price Distribution')
ax[0].set_xlabel('Sale Price')
ax[0].set_ylabel('Frequency')

ax[1].hist(np.log1p(finalTrain_data['SalePrice']), color='salmon', edgecolor='black')
ax[1].set_title('Log Transformed Sale Price Distribution')
ax[1].set_xlabel('Log(Sale Price)')
ax[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

##### Using The Log Transformed SalePrice Data

In [ ]:
log_data = finalTrain_data.copy()

In [ ]:
categorical_cols = log_data.select_dtypes(include=["object"]).columns
numeric_cols = log_data.select_dtypes(include=["int64", "int32","float64"]).columns
numeric_cols = numeric_cols.drop(["SalePrice", "SalesPrice", "Id"])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

##### RandomForest Model Using Log Transformed Sales Price

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model.fit(X_train, y_train)
rf_train_score = rf_model.score(X_train, y_train)
rf_test_score = rf_model.score(X_test, y_test)
print(f'The train score is :{rf_train_score:.3f}, and the test score is :{rf_test_score:.3f}')

In [ ]:
y_train_pred = rf_model.predict(X_train)
y_test_pred  = rf_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

##### XGBoost Model Using Log Transformed Sales Price

In [ ]:
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
xgb_model.fit(X_train, y_train)
xgb_train_score = xgb_model.score(X_train, y_train)
xgb_test_score = xgb_model.score(X_test, y_test)
print(f'The train score is :{xgb_train_score:.3f}, and the test score is :{xgb_test_score:.3f}')

In [ ]:
y_train_pred = xgb_model.predict(X_train)
y_test_pred  = xgb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

##### LightGBM Model Using Log Transformed Sales Price

In [ ]:
lgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=22
))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lgb_model.fit(X_train, y_train)
lgb_train_score = lgb_model.score(X_train, y_train)
lgb_test_score = lgb_model.score(X_test, y_test)
print(f'The train score is :{lgb_train_score:.3f}, and the test score is :{lgb_test_score:.3f}')

In [ ]:
y_train_pred = lgb_model.predict(X_train)
y_test_pred  = lgb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

##### CatBoost Model Using The Log Transformed Sales Price

In [ ]:
ctb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", ctb.CatBoostRegressor(
    iterations=100,  # Number of trees
    learning_rate=0.1,
    depth=3,
    verbose=200,  
    random_state=22
))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
ctb_model.fit(X_train, y_train)
ctb_train_score = ctb_model.score(X_train, y_train)
ctb_test_score = ctb_model.score(X_test, y_test)
print(f'The train score is :{ctb_train_score:.3f}, and the test score is :{ctb_test_score:.3f}')

In [ ]:
y_train_pred = ctb_model.predict(X_train)
y_test_pred  = ctb_model.predict(X_test)

Top_train = pd.DataFrame({"actual": y_train, "prediction": y_train_pred})
Top_test  = pd.DataFrame({"actual": y_test,  "prediction": y_test_pred})

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

In [ ]:
sns.regplot(x=y_test, y=y_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

#### Cross Validation And Hyperparameter Tunning

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

In [ ]:
X_train = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y_train = log_data["SalesPrice"]

##### Random Forest Model Training

In [ ]:
def objective(trial):

    rf = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators",100,500,step=100),
        max_depth=trial.suggest_int("max_depth",5,30,step=5),
        min_samples_split=trial.suggest_int("min_samples_split",2,10),
        min_samples_leaf=trial.suggest_int("min_samples_leaf",1,5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        bootstrap=trial.suggest_categorical("bootstrap", [True, False]),
        random_state=22,
        n_jobs=-1
    )

    model = Pipeline([
        ("preprocess", preprocessor),
        ("model", rf)
    ])

    score = cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()

    return score

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=1800)  # 10 trials OR max 30 mins

print("Best parameters:", study.best_params)

In [ ]:
best_params = study.best_params
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(**best_params, random_state=42))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model.fit(X_train, y_train)
cv_rf_train_score = rf_model.score(X_train, y_train)
cv_rf_test_score = rf_model.score(X_test, y_test)
print(f'The train score is :{cv_rf_train_score:.3f}, and the test score is :{cv_rf_test_score:.3f}')

In [ ]:
rf_cv_y_train_pred = rf_model.predict(X_train)
rf_cv_y_test_pred  = rf_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, rf_cv_y_test_pred))
mae = mean_absolute_error(y_test, rf_cv_y_test_pred)
rf_r2 = r2_score(y_test, rf_cv_y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", rf_r2)

##### LightGBM Model Training 

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
def objective(trial):

    model_lgb = LGBMRegressor(
        n_estimators=trial.suggest_int("n_estimators", 200, 1000, step=100),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        num_leaves=trial.suggest_int("num_leaves", 20, 200),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 50),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 0.0, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 0.0, 1.0),
        random_state=22,
        n_jobs=-1
    )

    model = Pipeline([
        ("preprocess", preprocessor),
        ("model", model_lgb)
    ])

    score = cross_val_score(model, X_train, y_train, cv=5, scoring="r2").mean()

    return score

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=1800)  # 50 trials OR max 30 mins

print("Best parameters:", study.best_params)

In [ ]:
best_params = study.best_params
lgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(**best_params, random_state=42))
])

In [ ]:
X = log_data.drop(["SalePrice", "SalesPrice", "Id"], axis=1)
y = log_data["SalesPrice"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
lgb_model.fit(X_train, y_train)
cv_lgb_train_score = lgb_model.score(X_train, y_train)
cv_lgb_test_score = lgb_model.score(X_test, y_test)
print(f'The train score is :{cv_lgb_train_score:.3f}, and the test score is :{cv_lgb_test_score:.3f}')

In [ ]:
lgb_cv_y_train_pred = lgb_model.predict(X_train)
lgb_cv_y_test_pred  = lgb_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, lgb_cv_y_test_pred))
mae = mean_absolute_error(y_test, lgb_cv_y_test_pred)
lgb_r2 = r2_score(y_test, lgb_cv_y_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", lgb_r2)

In [ ]:
feature_names = lgb_model.named_steps["preprocessor"].get_feature_names_out()

importances = lgb_model.named_steps["model"].feature_importances_

importances_df = (pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False))

importances_df.head(20)

##### Training The Model with Features actually Contributing

In [ ]:
important_features = importances_df[importances_df["importance"] > 5]["feature"]
print(len(important_features))

In [ ]:
X_train_processed = lgb_model.named_steps["preprocessor"].transform(X)

In [ ]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X.index
)

In [ ]:
X_train_reduced = X_train_processed[important_features]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_train_reduced, y, test_size=0.2, random_state=42
)

In [ ]:
final_model = LGBMRegressor(**study.best_params, random_state=22)

final_model.fit(X_train, y_train)

In [ ]:
lgb_train_score = final_model.score(X_train, y_train)
lgb_test_score = final_model.score(X_test, y_test)
print(f'The train score is :{lgb_train_score:.3f}, and the test score is :{lgb_test_score:.3f}')

In [ ]:
final_train_pred = final_model.predict(X_train)
final_test_pred  = final_model.predict(X_test)

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, final_test_pred))
mae = mean_absolute_error(y_test, final_test_pred)
final_r2 = r2_score(y_test, final_test_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", final_r2)

In [ ]:
sns.regplot(x=y_test, y=final_test_pred,
            scatter_kws={"alpha":0.6},
            line_kws={"color":"black"});

In [ ]:
fl_train_pred = np.expm1(final_train_pred)
fl_train = np.expm1(y_train)

In [ ]:
final_train_data  = pd.DataFrame({"actual": fl_train,  "prediction": fl_train_pred})

In [ ]:
final_train_data.head(7)

In [ ]:
fl_test_pred = np.expm1(final_test_pred)
fl_test = np.expm1(y_test)

In [ ]:
final_test  = pd.DataFrame({"actual": fl_test,  "prediction": fl_test_pred})

In [ ]:
final_test.head(7)

##### Retraining The Model Using The Full Train Data

In [ ]:
from sklearn.feature_selection import SelectFromModel

In [ ]:
final_model = LGBMRegressor(**study.best_params, random_state=22)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("feature_selection", SelectFromModel(final_model, threshold=0)),
    ("model", final_model)
])

pipeline.fit(X, y)

In [ ]:
complete_score = pipeline.score(X, y)
print(f'The train score is :{complete_score:.3f}')

### Importing the Test Data

In [ ]:
test_foder_path = 'C:/Users/Ryzen7/Downloads/House competition/test.csv'

In [ ]:
test_data = pd.read_csv(test_foder_path)

In [ ]:
test_data.info()

##### Cleaning the test data, exactly same method with the train data

In [ ]:
test_data['MiscFeature'].fillna('None', inplace = True)
test_data['Fence'].fillna('None', inplace = True)
test_data['PoolQC'].fillna('None', inplace = True)
test_data['GarageCond'].fillna('None', inplace = True)
test_data['GarageQual'].fillna('None', inplace = True)
test_data['GarageFinish'].fillna('None', inplace = True)
test_data['GarageType'].fillna('None', inplace = True)
test_data['FireplaceQu'].fillna('None', inplace = True)
test_data['BsmtFinType2'].fillna('None', inplace = True)
test_data['BsmtFinType1'].fillna('None', inplace = True)
test_data['BsmtExposure'].fillna('None', inplace = True)
test_data['BsmtCond'].fillna('None', inplace = True)
test_data['BsmtQual'].fillna('None', inplace = True)
test_data['MasVnrType'].fillna('None', inplace = True)
test_data['Alley'].fillna('None', inplace = True)
test_data['GarageYrBlt'].fillna(0, inplace = True)
test_data['HasGarage'] = test_data['GarageYrBlt'].apply(
    lambda x: 1 if x > 0 else 0
)
test_data['MasVnrArea'].fillna(0, inplace = True)
test_data['HasMansonry'] = (test_data['MasVnrArea'] > 0).astype(int)

In [ ]:
test_data['MSZoning'].fillna(test_data['MSZoning'].mode()[0], inplace = True)
test_data['Utilities'].fillna(test_data['Utilities'].mode()[0], inplace = True)
test_data['Exterior1st'].fillna('Other', inplace = True)
test_data['Exterior2nd'].fillna('Other', inplace = True)
test_data['KitchenQual'].fillna(test_data['KitchenQual'].mode()[0], inplace = True)
test_data['Functional'].fillna(test_data['Functional'].mode()[0], inplace = True)
test_data['SaleType'].fillna('Oth', inplace = True)

##### Same Feature Engineering 

In [ ]:
test_data['TotalSquareFootage'] = test_data['1stFlrSF'] + test_data['2ndFlrSF'] + test_data['TotalBsmtSF']

In [ ]:
test_data = test_data.rename(columns = {"1stFlrSF" : "FirstFlrSF",
                            "2ndFlrSF" : "SecondFlrSF",
                            "3SsnPorch" : "ThreeSsnPorch"
                            });

In [ ]:
test_data['LotShape'].value_counts()

In [ ]:
LotShape_mapping = {'Reg' : 0, 'IR1' : 1, 'IR2' : 2, 'IR3' : 3}

In [ ]:
test_data['LotShape'] = test_data['LotShape'].map(LotShape_mapping)

In [ ]:
test_data['LotShape'].value_counts()

In [ ]:
LandSlope_mapping = {'Gtl' : 0, 'Mod' : 1, 'Sev' : 2}
test_data['LandSlope'] = test_data['LandSlope'].map(LandSlope_mapping)
ExterQual_mapping =  {'Po' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}
test_data['ExterQual'] = test_data['ExterQual'].map(ExterQual_mapping )
ExterCond_mapping = {'Po' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}
test_data['ExterCond'] = test_data['ExterCond'].map(ExterCond_mapping)
BsmtQual_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}
test_data['BsmtQual'] = test_data['BsmtQual'].map(BsmtQual_mapping)
BsmtCond_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}
test_data['BsmtCond'] = test_data['BsmtCond'].map(BsmtCond_mapping)
BsmtExposure_mapping = {'None' : 0, 'No' : 1, 'Mn' : 2, 'Av' : 3, 'Gd' : 4}
test_data['BsmtExposure'] = test_data['BsmtExposure'].map(BsmtExposure_mapping)
BsmtFinType1_mapping = {'None':0, 'Unf':1, 'LwQ':2, 'Rec':3, 'BLQ':4, 'ALQ':5, 'GLQ':6}
test_data['BsmtFinType1'] = test_data['BsmtFinType1'].map(BsmtFinType1_mapping)
BsmtFinType2_mapping = {'GLQ':6, 'ALQ':5, 'BLQ':4, 'Rec':3, 'LwQ':2, 'Unf':1, 'None':0}
test_data['BsmtFinType2'] = test_data['BsmtFinType2'].map(BsmtFinType2_mapping)
HeatingQC_mapping = {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1, 'Po':0}
test_data['HeatingQC'] = test_data['HeatingQC'].map(HeatingQC_mapping)
CentralAir_mapping = { 'Y':1, 'N':0}
test_data['CentralAir'] = test_data['CentralAir'].map(CentralAir_mapping)
KitchenQual_mapping = {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1, 'Po':0}
test_data['KitchenQual'] = test_data['KitchenQual'].map(KitchenQual_mapping)
FireplaceQu_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}
test_data['FireplaceQu'] = test_data['FireplaceQu'].map(FireplaceQu_mapping)
GarageFinish_mapping = {'None':0, 'Unf':1, 'RFn':2, 'Fin':3}
test_data['GarageFinish'] = test_data['GarageFinish'].map(GarageFinish_mapping)
GarageQual_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}
test_data['GarageQual'] = test_data['GarageQual'].map(GarageQual_mapping)
GarageCond_mapping = {'None' : 0, 'Po' : 1, 'Fa' : 2, 'TA' : 3, 'Gd' : 4, 'Ex' : 5}
test_data['GarageCond'] = test_data['GarageCond'].map(GarageCond_mapping)
PavedDrive_mapping = { 'N':0, 'P':1, 'Y':2}
test_data['PavedDrive'] = test_data['PavedDrive'].map(PavedDrive_mapping)
PoolQC_mapping = {'None' : 0, 'Fa' : 1, 'TA' : 2, 'Gd' : 3, 'Ex' : 4}
test_data['PoolQC'] = test_data['PoolQC'].map(PoolQC_mapping)
Fence_mapping = {'None':0, 'MnWw':1, 'GdWo':2, 'MnPrv':3, 'GdPrv':4}
test_data['Fence'] = test_data['Fence'].map(Fence_mapping)

In [ ]:
test_data['HouseAge'] = train_data['YrSold'] - train_data['YearBuilt']

In [ ]:
test_data.info()

In [ ]:
test_data['PavedDrive'].value_counts()

In [ ]:
test_data.select_dtypes(include=["int64", "int32", "float64"]).isnull().sum().sort_values(ascending=False)

##### Assessing the test column matches with the trained model columns 

In [ ]:
set(finalTrain_data.columns) - set(test_data.columns)

In [ ]:
set(test_data.columns) - set(finalTrain_data.columns)

In [ ]:
num_cols = train_data.select_dtypes(include=["int64","int32","float64"]).columns

for col in num_cols:
    if col in test_data.columns:
        if test_data[col].isnull().sum() > 0:
            test_data[col] = test_data[col].fillna(finalTrain_data[col].median())

##### Prediction on the test data 

In [ ]:
final_predictions = pipeline.predict(test_data)

In [ ]:
final_predictions = np.expm1(final_predictions)

In [ ]:
submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": final_predictions
})

In [ ]:
print(submission.head(10))

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.to_csv('C:/Users/Ryzen7/Downloads/House competition/submission.csv', index=False)